In [22]:
import numpy as np

from numba import njit

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.preprocessing import StandardScaler

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import fastplotlib as fpl

import optuna

from dysts.maps import Henon

# Init

## Init Reservoir

In [34]:
steps = 20000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [35]:
henon_model = Henon()
henon_dataset = henon_model.make_trajectory(total_steps)
henon_dataset = henon_dataset[transient_steps_chaos:]

In [36]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

## Init Funcs

In [37]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.ndim == 1:
            fig.add_trace(
                go.Scatter(
                    x=np.arange(len(data)),
                    y=data,
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        elif data.ndim == 2:
            fig.add_trace(
                go.Scatter(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scatter3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [38]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    actual_list = actual_list.reshape(-1, 1) if actual_list.ndim == 1 else actual_list
    predicted_list = predicted_list.reshape(-1, 1) if predicted_list.ndim == 1 else predicted_list

    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [39]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [170]:
def weight_plot(weights, dims):
    dim_names = ["x", "y", "z"][:dims]
    num_features_per_type = len(weights) // 2
    num_nodes = num_features_per_type // dims

    labels = []
    for n in range(num_nodes):
        for d in range(dims):
            labels.append(f"Node {n + 1} {dim_names[d]} Pos")
            labels.append(f"Node {n + 1} {dim_names[d]} Vel")

    weights_flat = weights.flatten()
    pos_part = weights_flat[:num_features_per_type]
    vel_part = weights_flat[num_features_per_type:]

    combined_weights = np.empty_like(weights_flat)
    combined_weights[0::2] = pos_part
    combined_weights[1::2] = vel_part

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=combined_weights,
                marker_color=np.where(combined_weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Node State & Dimension",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [41]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=10,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    wall_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]

    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"
    if wall_nodes is not None and wall_nodes[0] != -1:
        node_colors[wall_nodes] = "blue"

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")
    coords = nodes_pos_3d + disp_3d[0]

    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )
    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

## Calc Init

In [42]:
@njit(cache=True)
def get_spring_forces(connections_list, disp, initial_pos, rest_lens, k_vals, num_nodes, dims):
    forces = np.zeros((num_nodes, dims))
    disp_reshaped = disp.reshape(num_nodes, dims)

    for i in range(len(connections_list)):
        idx_a = connections_list[i, 0]
        idx_b = connections_list[i, 1]

        delta = np.zeros(dims)
        dist_sq = 0.0
        for j in range(dims):
            pos_a = initial_pos[idx_a, j] + disp_reshaped[idx_a, j]
            pos_b = initial_pos[idx_b, j] + disp_reshaped[idx_b, j]
            delta[j] = pos_b - pos_a
            dist_sq += delta[j] ** 2

        dist = np.sqrt(dist_sq)

        mag = k_vals[i] * (dist - rest_lens[i])

        for j in range(dims):
            f_component = mag * (delta[j] / dist)
            forces[idx_a, j] += f_component
            forces[idx_b, j] -= f_component

    return forces.reshape(-1)

In [43]:
@njit(cache=True)
def run_simulation(
    steps, dt, m_inv_diag, c_diag, U, initial_pos, connections_list, k_vals, rest_lens, wall_nodes=[-1]
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    mask = np.ones(matrix_size)
    if wall_nodes[0] != -1:
        for wall in wall_nodes:
            idx = wall * dims
            mask[idx : idx + dims] = 0

    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )

    for i in range(1, steps):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])
        acc *= mask

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + .5 * acc * dt) + U[i])
        acc_next *= mask

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

# Non Lin

In [274]:
N = 100
dist_between = 2.5

x = np.zeros(N * 3)
for i in range(0, N):
    x[i * 3 : (i + 1) * 3] = np.array([0, 0, 0]) + dist_between * i
y = np.tile(np.array([0, 1, 2]), N)

nodes_pos = np.column_stack((x, y))

In [288]:
tau_steps = 1
free_steps = 3
dt = 0.01
input_force = 10
m_val = .01
c_val = 0.3
k_val = 10
k_between_val = 3
rest_lens_val = 1.1


N_step = int(N / 5)
target_nodes = 1 + np.array([N_step, 2 * N_step, 3 * N_step, 4 * N_step]) * 3

starts = 3 * np.arange(N)
wall_nodes = np.column_stack([starts, starts + 2]).flatten()

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_nodes = np.ones(num_nodes) * m_val
m_diag = np.repeat(m_nodes, dims)
m_inv_diag = 1.0 / m_diag

c_nodes = np.ones(num_nodes) * c_val
c_diag = np.repeat(c_nodes, dims)

In [289]:
node_ids = np.arange(x.size)

starts = 3 * np.arange(N)
connection_src = np.column_stack([starts, starts + 1]).flatten()
connection_dst = np.column_stack([starts + 1, starts + 2]).flatten()

betweens = np.arange(1, node_ids[-1], 3)
between_src = betweens[:-1]
between_dst = betweens[1:]

src_nodes = np.concatenate([connection_src, between_src])
dst_nodes = np.concatenate([connection_dst, between_dst])
connections_list = np.column_stack((src_nodes, dst_nodes))


k_connection_vals = np.ones(connection_src.shape[0]) * k_val
k_between_vals = np.ones(between_src.shape[0]) * k_between_val
k_vals = np.concatenate([k_connection_vals, k_between_vals])

rest_lens = np.ones(src_nodes.shape[0]) * rest_lens_val

In [290]:
total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

U = np.zeros((total_steps_with_free, matrix_size))
for i, node_index in enumerate(target_nodes):
    U[::free_steps, node_index * dims] = henon_scaled[:, i % dims]
# U = U * input_force  # U[0, target_nodes[0] * dims] = 1e

U = U * input_force

In [291]:
displacement, velocity = run_simulation(
    steps=total_steps_with_free,
    dt=dt,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    rest_lens=rest_lens,
    wall_nodes=wall_nodes,
)

In [279]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=10,
).show()

In [292]:
movement_nodes = 1 + np.arange(N) * 3
movement_idx = movement_nodes * dims
X = np.column_stack(
    (displacement[:, movement_idx], velocity[:, movement_idx])
)

X_sampled = X[free_steps - 1 :: free_steps]
X_delayed = X_sampled[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]

X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)
Y_train, Y_test = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

alphas=np.logspace(-5, 5, 30)
model = RidgeCV(alphas=alphas, cv=3)
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test)

In [293]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

print(r_2, mse)

0.6508804749662433 0.3057414792481919


In [294]:
weight_plot(np.linalg.norm(model.coef_, axis=0), 1).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# Non Lin

In [437]:
N = 100
dist_between = 2.5

x = np.zeros(N * 3)
for i in range(0, N):
    x[i * 3 : (i + 1) * 3] = np.array([0, 0, 0]) + dist_between * i
y = np.tile(np.array([0, 1, 2]), N)

nodes_pos = np.column_stack((x, y))

In [489]:
rng = np.random.default_rng(42)
sigma = 1

tau_steps = 1
free_steps = 3
dt = 0.01
input_force = 10
m_val = .01
c_val = 0.2
k_val = 5
k_between_val = 4
rest_lens_val = 1.1

target_node_count = 10
N_step = int(N / target_node_count)
target_nodes = 1 + (np.arange(target_node_count) * N_step) * 3

starts = 3 * np.arange(N)
wall_nodes = np.column_stack([starts, starts + 2]).flatten()

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

# m_nodes = np.ones(num_nodes) * m_val
mu = np.log(m_val) - (sigma**2 / 2)
m_nodes = rng.lognormal(mean=mu, sigma=sigma, size=num_nodes)
m_nodes = np.clip(m_nodes, a_min=.001, a_max=None)
m_diag = np.repeat(m_nodes, dims)
m_inv_diag = 1.0 / m_diag

mu = np.log(c_val) - (sigma**2 / 2)
c_nodes = rng.lognormal(mean=mu, sigma=sigma, size=num_nodes)
c_nodes = np.clip(c_nodes, a_min=0.001, a_max=None)
c_diag = np.repeat(c_nodes, dims)

In [490]:
node_ids = np.arange(x.size)

starts = 3 * np.arange(N)
connection_src = np.column_stack([starts, starts + 1]).flatten()
connection_dst = np.column_stack([starts + 1, starts + 2]).flatten()

betweens = np.arange(1, node_ids[-1], 3)
between_src = betweens[:-1]
between_dst = betweens[1:]

src_nodes = np.concatenate([connection_src, between_src])
dst_nodes = np.concatenate([connection_dst, between_dst])
connections_list = np.column_stack((src_nodes, dst_nodes))

rng = np.random.default_rng(42)

mu = np.log(k_val) - (sigma**2 / 2)
k_connection_vals = rng.lognormal(mean=mu, sigma=sigma, size=connection_src.shape[0]//2).repeat(2)
mu = np.log(k_between_val) - (sigma**2 / 2)
k_between_vals = rng.lognormal(mean=mu, sigma=sigma, size=between_src.shape[0])
k_vals = np.concatenate([k_connection_vals, k_between_vals])

mu = np.log(rest_lens_val) - (sigma**2 / 2)
rest_lens = rng.lognormal(
    mean=mu, sigma=sigma, size=connection_src.shape[0] // 2
).repeat(2)
rest_lens = np.clip(rest_lens, a_min=None, a_max=2)

In [491]:
total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

U = np.zeros((total_steps_with_free, matrix_size))
for i, node_index in enumerate(target_nodes):
    U[::free_steps, node_index * dims] = henon_scaled[:, i % dims]
# U = U * input_force  # U[0, target_nodes[0] * dims] = 1e

U = U * input_force

In [492]:
displacement, velocity = run_simulation(
    steps=total_steps_with_free,
    dt=dt,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    rest_lens=rest_lens,
    wall_nodes=wall_nodes,
)

In [ ]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
).show()

/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_5481/1061758632.py:80: RuntimeWarning: overflow encountered in cast
  dots.data = coords.astype(np.float32)
/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_5481/1061758632.py:84: RuntimeWarning: overflow encountered in cast
  l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)


: 

In [434]:
movement_nodes = 1 + np.arange(N) * 3
movement_idx = movement_nodes * dims
X = np.column_stack(
    (displacement[:, movement_idx], velocity[:, movement_idx])
)

X_sampled = X[free_steps - 1 :: free_steps]
X_delayed = X_sampled[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]

X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)
Y_train, Y_test = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

alphas=np.logspace(-5, 5, 30)
model = RidgeCV(alphas=alphas, cv=3)
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test)

In [435]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

print(r_2, mse)

0.8990793041713756 0.16334867946068873


In [436]:
weight_plot(np.linalg.norm(model.coef_, axis=0), 1).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()